# 02 - Pitch archetypes

**Goal:** group pitches into archetypes so park effects can be measured on groups with enough data, instead of on individual pitchers.

An archetype is a pitch group (four-seam, sinker, slider, ...) crossed with percentile tiers **within that pitch group** for arm slot, velocity, and induced vertical break. Tier counts are set in `src/archetypes.py` and are easy to change. The default is 2 x 2 x 2 = 8 archetypes per pitch group. More tiers make archetypes more specific but leave fewer balls in play per park, so check the cell sizes in notebook 03 before adding tiers.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
from src.config import PROCESSED, INTERIM
from src.data import load_seasons, add_park, ANALYSIS_COLS
from src.archetypes import add_shape_features, add_archetypes

In [ ]:
df = add_park(load_seasons(columns=ANALYSIS_COLS))
df = add_shape_features(df)
df = add_archetypes(df)
df = df.dropna(subset=["archetype"])
print(f"{len(df):,} pitches, {df['archetype'].nunique()} archetypes")

## Sanity check: movement direction

Arm-side run is positive for both hands after the sign flip. Sinkers and changeups should show positive values for both RHP and LHP, and sliders, sweepers, and cutters should be negative. If not, the sign convention in `add_shape_features` needs fixing before anything else.

In [ ]:
df.groupby(["pitch_group", "p_throws"])["arm_side_in"].mean().round(1).unstack()

## Archetype sizes

In [ ]:
sizes = df["archetype"].value_counts()
sizes.describe()

In [ ]:
ff = df[df["pitch_group"] == "Four-seam"]
ff = ff.sample(min(20000, len(ff)), random_state=0)

fig, ax = plt.subplots(figsize=(7, 6))
for tier, g in ff.groupby("slot_tier"):
    ax.scatter(g["arm_side_in"], g["ivb_in"], s=3, alpha=0.3, label=f"{tier} slot")
ax.set_xlabel("Arm-side run (in)")
ax.set_ylabel("Induced vertical break (in)")
ax.set_title("Four-seam fastball shape by arm-slot tier")
ax.legend(markerscale=4)
plt.show()

## Save cut points and outputs

The cut points document exactly what "high slot" or "faster" means for each pitch group. Put them on the methodology slide.

In [ ]:
cut = (df.groupby("pitch_group")[["release_speed", "ivb_in", "slot_value"]]
         .quantile([1/3, 0.5, 2/3]).round(1))
cut.to_csv(PROCESSED / "archetype_cutpoints.csv")
cut

In [ ]:
INTERIM.mkdir(parents=True, exist_ok=True)
df.to_parquet(INTERIM / "pitches_archetyped.parquet", index=False)
sizes.rename_axis("archetype").reset_index(name="pitches").to_csv(PROCESSED / "archetype_sizes.csv", index=False)

## Commit

```
git add notebooks/02_archetypes.ipynb data/processed/archetype_cutpoints.csv data/processed/archetype_sizes.csv
git commit -m "define pitch archetypes and save cut points"
git push
```